# Kronos Crypto Price Trend Prediction - Example Usage

This notebook demonstrates how to use the Kronos crypto prediction system for:
1. Data ingestion
2. Feature engineering
3. Model training
4. Backtesting
5. Hyperparameter tuning
6. Generating reports

## Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.insert(0, str(Path('..') / 'src'))

from utils import load_config, setup_logging
from data_ingest import BinanceDataFetcher
from labeling import TrendLabeler
from features import FeatureGenerator
from model_kronos import KronosModel, ModelTrainer
from backtest import WalkForwardBacktester
from tune import HyperparameterTuner
from report import ReportGenerator

# Setup plotting
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Load Configuration

In [ ]:
# Load config
config_path = Path('../configs/config.yaml')
config = load_config(config_path)

# Setup logging
logger = setup_logging(
    log_level="INFO",
    log_dir=Path("../logs"),
    log_to_file=False,
    log_to_console=True
)

print(f"Configuration loaded successfully")
print(f"Symbols: {config['data']['symbols']}")
print(f"Lookback days: {config['data']['lookback_days']}")
print(f"Prediction horizon: {config['labeling']['horizon_minutes']} minutes")

## 1. Data Ingestion

Download historical price data from Binance

In [ ]:
# Create data fetcher
fetcher = BinanceDataFetcher(config)

# Test connection
success, msg = fetcher.test_connection()
print(f"Connection test: {msg}")

# Fetch data for one symbol (BTCUSDT)
from datetime import datetime, timedelta

symbol = "BTCUSDT"
end_date = datetime.utcnow()
start_date = end_date - timedelta(days=30)  # Last 30 days for demo

df = fetcher.fetch_historical_data(symbol, start_date, end_date)
print(f"\nFetched {len(df)} records for {symbol}")
df.head()

### Visualize Price Data

In [ ]:
# Plot price
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Price
ax1.plot(df['open_time'], df['close'], linewidth=0.5)
ax1.set_title(f'{symbol} Close Price')
ax1.set_xlabel('Date')
ax1.set_ylabel('Price (USD)')
ax1.grid(True, alpha=0.3)

# Volume
ax2.bar(df['open_time'], df['volume'], width=0.0007, alpha=0.5)
ax2.set_title(f'{symbol} Volume')
ax2.set_xlabel('Date')
ax2.set_ylabel('Volume')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Basic statistics
print("\nPrice Statistics:")
print(df['close'].describe())

## 2. Labeling

Create trend labels for 10-minute predictions

In [ ]:
# Create labeler
labeler = TrendLabeler(config)

# Generate labels
df_labeled = labeler.create_labels(df.copy())

# Get distribution
distribution = labeler.get_label_distribution(df_labeled)
print("\nLabel Distribution:")
for label, stats in distribution.items():
    print(f"  {label}: {stats['count']} ({stats['percentage']:.2f}%)")

# Visualize label distribution
labels = list(distribution.keys())
counts = [distribution[l]['count'] for l in labels]

plt.figure(figsize=(8, 5))
plt.bar(labels, counts, color=['red', 'gray', 'green'])
plt.title('Label Distribution')
plt.ylabel('Count')
plt.xlabel('Label')
for i, count in enumerate(counts):
    pct = distribution[labels[i]]['percentage']
    plt.text(i, count, f"{pct:.1f}%", ha='center', va='bottom')
plt.show()

## 3. Feature Engineering

Generate 50+ technical indicators

In [ ]:
# Create feature generator
feature_gen = FeatureGenerator(config)

# Generate features
df_features = feature_gen.generate_all_features(df_labeled)

# Get feature columns
feature_cols = feature_gen.get_feature_columns(df_features)
print(f"\nGenerated {len(feature_cols)} features")
print(f"\nSample features: {feature_cols[:10]}")

# Validate features
is_valid = feature_gen.validate_features(df_features)
print(f"\nFeatures valid: {is_valid}")

df_features.head()

### Visualize Sample Features

In [ ]:
# Plot some interesting features
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
axes = axes.flatten()

features_to_plot = [
    'log_return_10m', 'rsi_14', 'macd', 
    'atr_14m', 'volume_zscore_30m', 'rolling_std_30m'
]

for i, feature in enumerate(features_to_plot):
    if feature in df_features.columns:
        axes[i].plot(df_features['open_time'], df_features[feature], linewidth=0.5)
        axes[i].set_title(feature)
        axes[i].grid(True, alpha=0.3)
        axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Model Training

Train a simple model on train/test split

In [ ]:
from sklearn.model_selection import train_test_split

# Prepare features and labels
X = df_features[feature_cols]
y = df_features['label'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=config['seed'], stratify=y
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

# Create and train model
model = KronosModel(config)
metrics = model.train(X_train, y_train, X_test, y_test)

# Print metrics
print("\nTraining Metrics:")
for key, value in metrics.items():
    if not isinstance(value, np.ndarray):
        print(f"  {key}: {value:.4f}")

### Feature Importance

In [ ]:
# Get feature importance
importance_df = model.get_feature_importance()

if importance_df is not None:
    # Plot top 20 features
    top_n = 20
    top_features = importance_df.head(top_n)
    
    plt.figure(figsize=(10, 8))
    plt.barh(range(top_n), top_features['importance'])
    plt.yticks(range(top_n), top_features['feature'])
    plt.gca().invert_yaxis()
    plt.xlabel('Importance')
    plt.title(f'Top {top_n} Feature Importance')
    plt.tight_layout()
    plt.show()
    
    print(f"\nTop 10 features:")
    print(top_features.head(10))

## 5. Walk-Forward Backtesting

Run time-series cross-validation

In [ ]:
# Note: This uses smaller windows for demo purposes
# Modify config for proper backtesting

backtester = WalkForwardBacktester(config)

# Run backtest (this may take a while)
results = backtester.backtest_symbol(
    symbol=symbol,
    df=df_features,
    save_results=False
)

# Display aggregated results
print("\nAggregated Results:")
for key, value in results['aggregated'].items():
    print(f"  {key}: {value:.4f}")

## 6. Hyperparameter Tuning (Quick Demo)

Find optimal hyperparameters (reduced trials for demo)

In [ ]:
# Create a config copy with fewer trials for demo
demo_config = config.copy()
demo_config['tuning']['n_trials'] = 10  # Reduced from 100

tuner = HyperparameterTuner(demo_config)

# Run tuning
results = tuner.tune_symbol_simple(
    symbol=symbol,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_test,
    y_valid=y_test
)

print(f"\nBest parameters:")
for key, value in results['best_params'].items():
    print(f"  {key}: {value}")
    
print(f"\nBest validation accuracy: {results['best_value']:.4f}")

## Summary

This notebook demonstrated the key components of the Kronos system:

1. **Data Ingestion**: Downloaded 30 days of BTC data from Binance
2. **Labeling**: Created trend labels for 10-minute predictions
3. **Feature Engineering**: Generated 50+ technical indicators
4. **Model Training**: Trained LightGBM model with class balancing
5. **Backtesting**: Performed walk-forward validation
6. **Hyperparameter Tuning**: Optimized model parameters with Optuna

For production use:
- Use `make all` to run the complete pipeline
- Increase training data to 730 days
- Run full hyperparameter search (100+ trials)
- Review generated reports in `reports/` directory